In [ ]:
# [Benchmark] Standardized Import & Setup
import os
import shutil
import datetime
import csv
import math
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    adjusted_rand_score, 
    normalized_mutual_info_score, 
    adjusted_mutual_info_score, 
    homogeneity_score, 
    completeness_score, 
    v_measure_score,
    f1_score,
    pairwise,
    silhouette_score
)
from scipy.optimize import linear_sum_assignment
from sklearn.neighbors import kneighbors_graph
import torch
import torch.nn.functional as F
import tracemalloc
from torch.nn import Linear
from torch_geometric.loader import DenseDataLoader
from torch_geometric.nn import DenseGraphConv, dense_mincut_pool
from torch_geometric.data import Data, InMemoryDataset
import torch_geometric.transforms as T
import warnings

# Global Settings
warnings.filterwarnings("ignore")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
sc.settings.verbosity = 0

In [ ]:
# [Utils] Standardization Functions (Plotting & Metrics)

def calculate_metrics(adata, pred_key, true_key, resource_stats, emb_key='latent_embedding'):
    """Calculate clustering metrics and output performance statistics."""
    metrics = {}
    
    # 1. Ground Truth Metrics
    if true_key in adata.obs.columns:
        labels_true = adata.obs[true_key]
        labels_pred = adata.obs[pred_key]
        
        # Standard Clustering Metrics
        metrics['ARI'] = adjusted_rand_score(labels_true, labels_pred)
        metrics['NMI'] = normalized_mutual_info_score(labels_true, labels_pred)
        metrics['AMI'] = adjusted_mutual_info_score(labels_true, labels_pred)
        metrics['Homogeneity'] = homogeneity_score(labels_true, labels_pred)
        metrics['Completeness'] = completeness_score(labels_true, labels_pred)
        metrics['V-Measure'] = v_measure_score(labels_true, labels_pred)

        # Optimal Matching for Classification Metrics (Macro-F1 & Composition Similarity)
        ct = pd.crosstab(labels_true, labels_pred)
        row_ind, col_ind = linear_sum_assignment(-ct.values)
        map_pred_to_true = {}
        for r, c in zip(row_ind, col_ind):
            map_pred_to_true[ct.columns[c]] = ct.index[r]
        mapped_preds = labels_pred.map(map_pred_to_true)
        if mapped_preds.isna().any():
            mapped_preds = mapped_preds.astype(object).fillna("Unmapped").astype(str)

        # Macro-F1 Score (using mapped labels)
        true_cats = labels_true.astype('category').cat.categories
        metrics['Macro-F1'] = f1_score(labels_true, mapped_preds, labels=true_cats, average='macro', zero_division=0)

        # Composition Similarity (using mapped labels)
        if 'cell_type' in adata.obs:
            gt_comp = pd.crosstab(adata.obs[true_key], adata.obs['cell_type'], normalize='index')
            pred_comp = pd.crosstab(mapped_preds, adata.obs['cell_type'], normalize='index')
            common_niches = gt_comp.index.intersection(pred_comp.index)
            if len(common_niches) > 0:
                gt_vecs = gt_comp.loc[common_niches].values
                pred_vecs = pred_comp.loc[common_niches].values
                cos_sims = np.diag(pairwise.cosine_similarity(gt_vecs, pred_vecs))
                metrics['Composition Similarity'] = np.mean(cos_sims)
            else:
                metrics['Composition Similarity'] = 0.0
        else:
             metrics['Composition Similarity'] = 0.0
    else:
        print("Warning: Ground truth key not found. Supervised metrics set to 0.")
        metrics = {k: 0.0 for k in ['ARI', 'NMI', 'AMI', 'Homogeneity', 'Completeness', 'V-Measure', 'Macro-F1', 'Composition Similarity']}
    
    # Spatial Connectivity
    try:
        if 'spatial_connectivities' not in adata.obsp:
            coords = adata.obsm['spatial'] if 'spatial' in adata.obsm else adata.obs[['x', 'y']].values
            adj = kneighbors_graph(coords, n_neighbors=6, mode='connectivity', include_self=False)
            adata.obsp['spatial_connectivities'] = adj
        W_conn = adata.obsp['spatial_connectivities']

        # Vectorized computation
        if hasattr(W_conn, 'indices'):
            # Convert to one-hot (N x K)
            y_onehot = pd.get_dummies(labels_pred).values
            # Neighbor counts per class: (N x N) @ (N x K) -> (N x K)
            neighbor_counts = W_conn @ y_onehot
            # Get count of neighbors with same label
            label_indices = np.argmax(y_onehot, axis=1)
            same_counts = neighbor_counts[np.arange(len(label_indices)), label_indices]
            
            degrees = np.array(W_conn.sum(axis=1)).flatten()
            mask = degrees > 0
            if mask.any():
                metrics['Connectivity'] = float(np.mean(same_counts[mask] / degrees[mask]))
            else:
                metrics['Connectivity'] = 0.0
        else:
            metrics['Connectivity'] = 0.0
    except Exception:
        metrics['Connectivity'] = 0.0

    # Silhouette Score (Latent Embedding)
    if emb_key in adata.obsm:
        try:
            label_key = true_key if true_key in adata.obs else pred_key
            metrics['Silhouette Score'] = silhouette_score(adata.obsm[emb_key], adata.obs[label_key])
        except Exception as e:
            print(f"Warning: Silhouette Score calculation failed: {e}")
            metrics['Silhouette Score'] = 0.0
    else:
        print(f"Warning: Latent embedding key '{emb_key}' not found in adata.obsm. Silhouette Score set to 0.")
        metrics['Silhouette Score'] = 0.0

    print(f"{'='*40}")
    print(f"BENCHMARK METRICS REPORT")
    print(f"{'='*40}")
    for k, v in metrics.items():
        print(f"{k:<20}: {v:.4f}")
        
    print(f"{'-'*40}")
    print(f"Time (s)            : {resource_stats['time']:.4f}")
    print(f"Memory (MB)         : {resource_stats['memory']:.4f}")
    print(f"Memory Peak (MB)    : {resource_stats['memory_peak']:.4f}")
    print(f"{'='*40}")
    
    return metrics

def plot_benchmark_results(adata, pred_key, true_key, cell_type_key, emb_key='latent_embedding'):
    """Standardized visualization: Ground Truth, Prediction, and Cell Type Composition."""
    
    for key in [pred_key, true_key, cell_type_key]:
        if key in adata.obs and not pd.api.types.is_categorical_dtype(adata.obs[key]):
            adata.obs[key] = adata.obs[key].astype('category')

    # A. Ground Truth Map
    print("Plotting Ground Truth Niche...")
    if true_key in adata.obs:
        if f'{true_key}_colors' in adata.uns:
            del adata.uns[f'{true_key}_colors']
            
        fig, ax = plt.subplots(figsize=(6, 6))
        sc.pl.scatter(adata, x='x', y='y', color=true_key, 
                     title='Ground Truth Niche', size=120000/adata.n_obs, frameon=False, show=False, ax=ax)
        ax.set_aspect('equal', 'box')
        plt.show()
    else:
        plt.figure(figsize=(6, 6))
        plt.text(0.5, 0.5, 'Ground Truth Not Available', ha='center', va='center')
        plt.axis('off')
        plt.show()

    # B. Predicted Map
    print("Plotting Predicted Niche...")
    if f'{pred_key}_colors' in adata.uns:
        del adata.uns[f'{pred_key}_colors']
        
    fig, ax = plt.subplots(figsize=(6, 6))
    sc.pl.scatter(adata, x='x', y='y', color=pred_key, 
                 title='Predicted Niche', size=120000/adata.n_obs, frameon=False, show=False, ax=ax)
    ax.set_aspect('equal', 'box')
    plt.show()

    # C. Latent Embedding UMAP
    print("Plotting Latent Embedding UMAP...")
    
    if emb_key in adata.obsm:
        sc.pp.neighbors(adata, use_rep=emb_key, n_neighbors=30)
        sc.tl.umap(adata)
        
        fig, ax = plt.subplots(figsize=(6, 6))
        sc.pl.umap(adata, color=pred_key, title=f'Latent Embedding UMAP', 
                   frameon=False, show=False, ax=ax)
        plt.show()
    else:
        print(f"Latent embedding key '{emb_key}' not found in adata.obsm")

    # D. Composition Analysis
    print("Plotting Composition Analysis...")
    
    if f'{cell_type_key}_colors' not in adata.uns:
        from scanpy.plotting import palettes
        length = len(adata.obs[cell_type_key].cat.categories)
        if length <= 20:
            palette = palettes.default_20
        else:
            palette = palettes.default_102
        adata.uns[f'{cell_type_key}_colors'] = palette[:length]
    
    ctype_colors = adata.uns[f'{cell_type_key}_colors']
    ctype_cats = adata.obs[cell_type_key].cat.categories
    ctype_cmap = dict(zip(ctype_cats, ctype_colors))

    df = pd.DataFrame({'pred': adata.obs[pred_key], 'cell_type': adata.obs[cell_type_key]})
    comp = pd.crosstab(df['pred'], df['cell_type'], normalize='index')
    
    bar_colors = [ctype_cmap.get(c, 'grey') for c in comp.columns]
    
    plt.figure(figsize=(10, 6))
    ax = plt.gca()
    comp.plot(kind='bar', stacked=True, width=0.8, ax=ax, color=bar_colors)
    
    ax.set_title('Cell Type Composition per Predicted Niche')
    ax.set_xlabel('Predicted Niche')
    ax.set_ylabel('Fraction')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., title='Cell Type')
    
    plt.tight_layout()
    plt.show()

def save_benchmark_results(adata, output_path, method_name, pred_key, emb_key):
    """
    Save benchmark results to a persistent h5ad file.
    Appends/Updates 'pred_niche_{method_name}' and 'X_{method_name}'.
    """
    print(f"[Save] Saving results for '{method_name}' to {output_path}...")
    
    target_pred = f"pred_niche_{method_name}"
    target_emb = f"X_{method_name}"
    
    if os.path.exists(output_path):
        print(f"  - Loading existing data from {output_path}...")
        adata_out = sc.read_h5ad(output_path)
    else:
        print(f"  - File not found. Creating new dataset from current results...")
        adata_out = adata.copy()       
        keys_to_keep = [k for k in adata_out.uns.keys() if 'colors' in k]
        adata_out.uns = {k: adata_out.uns[k] for k in keys_to_keep}
    
    if pred_key in adata.obs:
        print(f"  - Updating obs['{target_pred}']")
        adata_out.obs[target_pred] = adata.obs[pred_key]
    else:
        print(f"  - Warning: Prediction key '{pred_key}' not found in current adata.")

    if emb_key in adata.obsm:
        print(f"  - Updating obsm['{target_emb}']")
        if adata_out.shape[0] == adata.shape[0]:
            adata_out.obsm[target_emb] = adata.obsm[emb_key]
        else:
            print(f"  - Error: Shape mismatch for embedding (Out: {adata_out.shape}, In: {adata.shape}). Skipping embedding save.")
    else:
        print(f"  - Warning: Embedding key '{emb_key}' not found in current adata.")

    print(f"  - Writing to disk...")
    try:
        adata_out.write_h5ad(output_path)
        print(f"  - Save complete.")
        print(adata_out)
    except Exception as e:
        print(f"  - Error saving file: {e}")

def save_time_memory_results(stats, method_name, file_path='./time_memory.csv'):
    """Save time and memory stats to CSV, updating if method exists."""
    data = {
        'Method': method_name,
        'Time': stats['time'],
        'Memory': stats['memory'],
        'Memory_Peak': stats['memory_peak']
    }
    
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        if method_name in df['Method'].values:
            for key, value in data.items():
                if key != 'Method':
                    df.loc[df['Method'] == method_name, key] = value
        else:
            df = pd.concat([df, pd.DataFrame([data])], ignore_index=True)
    else:
        df = pd.DataFrame([data])
        
    df.to_csv(file_path, index=False)
    print(f"[Save] Resource stats saved to {file_path}")

In [ ]:
# [CytoCommunity] Core Classes & Functions

# 1. Dataset Class
class SpatialOmicsImageDataset(InMemoryDataset):
    def __init__(self, root, data_list=None, transform=None, pre_transform=None):
        self.data_list = data_list
        super(SpatialOmicsImageDataset, self).__init__(root, transform, pre_transform)
        self.data, self.slices = torch.load(self.processed_paths[0])

    @property
    def raw_file_names(self):
        return []

    @property
    def processed_file_names(self):
        return ['SpatialOmicsImageDataset.pt']

    def download(self):
        pass
    
    def process(self):
        # If data_list is provided (first run), save it
        if self.data_list:
            data, slices = self.collate(self.data_list)
            torch.save((data, slices), self.processed_paths[0])

# 2. Model Class 
class Net(torch.nn.Module):
    def __init__(self, in_channels, out_channels, num_tcn, hidden_channels=128):
        super(Net, self).__init__()
        self.conv1 = DenseGraphConv(in_channels, hidden_channels)
        self.pool1 = Linear(hidden_channels, num_tcn)

    def forward(self, x, adj, mask=None):
        x = F.relu(self.conv1(x, adj, mask))
        
        # --- Capture Latent Embedding ---
        embedding = x.clone() 
        
        s = self.pool1(x) 
        x, adj, mc1, o1 = dense_mincut_pool(x, adj, s, mask)
        ClusterAssignTensor_1 = s
        ClusterAdjTensor_1 = adj
        
        # Return embedding as the last element
        return F.log_softmax(x, dim=-1), mc1, o1, ClusterAssignTensor_1, ClusterAdjTensor_1, embedding

def train_model(model, optimizer, train_loader, device):
    model.train()
    loss_all = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        # Unpack 6 values now
        out, mc_loss, o_loss, _, _, _ = model(data.x, data.adj, data.mask)
        loss = mc_loss + o_loss
        loss.backward()
        loss_all += loss.item()
        optimizer.step()
    return loss_all

# 3. Main Wrapper
def run_cytocommunity_method(adata, image_name="sample_0", num_tcn=5, knn_k=30, num_run=5, num_epoch=500):
    """
    Executes the full CytoCommunity workflow: Data Prep -> Step 1 -> Step 2 -> Step 3 (R) -> Load Results.
    """
    # -- Setup Directories --
    work_dir = os.path.abspath(f"./CytoCommunity_Work/{image_name}")
    input_dir = os.path.join(work_dir, "Input")
    step1_dir = os.path.join(work_dir, "Step1_Output")
    step2_dir = os.path.join(work_dir, f"Step2_Output_{image_name}")
    step3_dir = os.path.join(work_dir, f"Step3_Output_{image_name}")
    
    for d in [input_dir, step1_dir, step2_dir, step3_dir]:
        if os.path.exists(d):
            shutil.rmtree(d)
        os.makedirs(d, exist_ok=True)
    
    start_time = datetime.datetime.now()
    tracemalloc.start()
    
    print("[CytoCommunity] Step 1: Data Preparation & Graph Construction...")
    
    # A. Export Data (Input)
    coords = adata.obs[['x', 'y']].values if 'x' in adata.obs else adata.obsm['spatial']
    if hasattr(coords, 'values'): coords = coords.values
    np.savetxt(os.path.join(input_dir, f"{image_name}_Coordinates.txt"), coords, delimiter="\t", fmt='%f')
    
    cell_types = adata.obs['cell_type'].values
    np.savetxt(os.path.join(input_dir, f"{image_name}_CellTypeLabel.txt"), cell_types, delimiter="\t", fmt='%s')
    
    # B. Construct Graph (Step 1 Logic)
    KNNgraph_sparse = kneighbors_graph(coords, knn_k, mode='connectivity', include_self=False, n_jobs=-1)
    KNNgraph_AdjMat = KNNgraph_sparse.toarray()
    KNNgraph_AdjMat_fix = KNNgraph_AdjMat + KNNgraph_AdjMat.T
    edge_ndarray = np.argwhere(KNNgraph_AdjMat_fix > 0)
    
    unique_cell_types = sorted(list(set(cell_types)))
    num_nodes = len(cell_types)
    node_attr_matrix = np.zeros((num_nodes, len(unique_cell_types)), dtype=np.float32)
    
    type_to_idx = {t: i for i, t in enumerate(unique_cell_types)}
    for i, t in enumerate(cell_types):
        node_attr_matrix[i, type_to_idx[t]] = 1.0
        
    # Create Data Object
    edge_index = torch.from_numpy(edge_ndarray.astype('int64'))
    x = torch.from_numpy(node_attr_matrix)
    data = Data(x=x, edge_index=edge_index.t().contiguous())
    
    # Create Dataset
    dataset = SpatialOmicsImageDataset(step1_dir, data_list=[data], transform=T.ToDense(num_nodes))

    print("[CytoCommunity] Step 2: TCN Learning (Unsupervised)...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    train_loader = DenseDataLoader(dataset, batch_size=1)
    latent_embedding = None
    
    for run_i in range(1, num_run + 1):
        print(f"  > Run {run_i}/{num_run}...")
        
        model = Net(dataset.num_features, 1, num_tcn=num_tcn).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
        
        run_folder = os.path.join(step2_dir, f"Run{run_i}")
        os.makedirs(run_folder, exist_ok=True)
        
        # Training
        prev_loss = float('inf')
        for epoch in range(1, num_epoch + 1):
            loss = train_model(model, optimizer, train_loader, device)
            if loss == 0 and loss == prev_loss:
                break
            prev_loss = loss
            
        model.eval()
        for data in train_loader:
            data = data.to(device)
            res = model(data.x, data.adj, data.mask)
            # Soft Assignment
            assign_mat = torch.softmax(res[3][0,:,:], dim=-1).detach().cpu().numpy()
            
            # Filter by mask (valid nodes)
            mask = data.mask.cpu().numpy().flatten()
            valid_assign = assign_mat[mask]
            
            if run_i == 1:
                emb_tensor = res[5][0,:,:]
                emb_np = emb_tensor.detach().cpu().numpy()
                latent_embedding = emb_np[mask]
            
            np.savetxt(os.path.join(run_folder, "TCN_AssignMatrix1.csv"), assign_mat, delimiter=',')
            np.savetxt(os.path.join(run_folder, "NodeMask.csv"), mask.astype(int), delimiter=',', fmt='%i')

    print("[CytoCommunity] Step 3: Ensemble (Majority Voting via R)...")
    r_script_path = os.path.join(work_dir, "run_ensemble.R")
    
    step2_dir_r = step2_dir.replace(os.sep, '/') + "/"
    step3_dir_r = step3_dir.replace(os.sep, '/') + "/"
    
    r_code = f"""
    library(diceR)
    
    step2_dir <- "{step2_dir_r}"
    step3_dir <- "{step3_dir_r}"
    
    if(!dir.exists(step3_dir)) dir.create(step3_dir, recursive=TRUE)
    
    NodeMask <- read.csv(paste0(step2_dir, "Run1/NodeMask.csv"), header = FALSE)
    nonzero_ind <- which(NodeMask$V1 == 1)
    
    allSoftClustFile <- list.files(path = step2_dir, pattern = "TCN_AssignMatrix1.csv", recursive = TRUE)
    allHardClustLabel <- vector()
    
    if(length(allSoftClustFile) > 0) {{
        for (i in 1:length(allSoftClustFile)) {{
          ClustMatrix <- read.csv(paste0(step2_dir, allSoftClustFile[i]), header = FALSE, sep = ",")
          ClustMatrix <- ClustMatrix[nonzero_ind,]
          HardClustLabel <- apply(as.matrix(ClustMatrix), 1, which.max)
          allHardClustLabel <- cbind(allHardClustLabel, as.vector(HardClustLabel))
        }}
        
        finalClass <- diceR::majority_voting(allHardClustLabel, is.relabelled = FALSE)
        write.table(finalClass, file = paste0(step3_dir, "TCNLabel_MajorityVoting.csv"), 
                    row.names = FALSE, col.names = FALSE, quote = FALSE)
    }}
    """
    with open(r_script_path, "w") as f:
        f.write(r_code)
        
    # Run R script
    ret = os.system(f"Rscript {r_script_path}")
    
    final_labels = None
    res_path = os.path.join(step3_dir, "TCNLabel_MajorityVoting.csv")
    
    if ret == 0 and os.path.exists(res_path):
        # Load Result
        final_labels = np.loadtxt(res_path, dtype=int)
    else:
        print("Warning: R execution failed or diceR not available. Using Run 1 as fallback.")
        assign = np.loadtxt(os.path.join(step2_dir, "Run1/TCN_AssignMatrix1.csv"), delimiter=',')
        mask_f = np.loadtxt(os.path.join(step2_dir, "Run1/NodeMask.csv"), delimiter=',')
        valid_assign = assign[mask_f == 1]
        final_labels = np.argmax(valid_assign, axis=1) + 1
        
    adata.obs['pred_niche'] = pd.Categorical(final_labels)
    if latent_embedding is not None:
        if latent_embedding.shape[0] == adata.n_obs:
            adata.obsm['latent_embedding'] = latent_embedding
        else:
            print(f"Warning: Embedding shape {latent_embedding.shape} mismatch with adata {adata.shape}")
    
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    end_time = datetime.datetime.now()
    
    stats = {
        "time": (end_time - start_time).total_seconds(),
        "memory": current / 10**6,
        "memory_peak": peak / 10**6,
        "epochs": num_epoch * num_run
    }
    
    return adata, stats

In [ ]:
# [Main] Execution

if __name__ == "__main__":
    tasks = [
        ("./lymph_node_niche_annotated.h5ad", "./lymph_node_niche_results.h5ad", "./time_memory.csv")
    ]
    N_CLUSTERS = 4
    
    for FILE_PATH, OUTPUT_PATH, TIMEMEM_PATH in tasks:
        print(f"\n[{'='*30} Processing {FILE_PATH} {'='*30}]")
        
        # 1. Load Data
        if not os.path.exists(FILE_PATH):
            print(f"Data file not found: {FILE_PATH}, skipping...")
            continue
        
        print(f"[Data] Loading {FILE_PATH}...")
        adata = sc.read_h5ad(FILE_PATH)
        
        if 'x' not in adata.obs.columns:
            if 'spatial' in adata.obsm:
                adata.obs['x'] = adata.obsm['spatial'][:, 0]
                adata.obs['y'] = adata.obsm['spatial'][:, 1]
            else:
                print("Warning: No spatial coordinates found.")

        print(f"[Data] Shape: {adata.shape}")
        if 'niche' in adata.obs:
            print(f"[Data] Ground Truth Niches: {len(adata.obs['niche'].unique())}")
        
    
        # 2. Run Method
        print(f"[Benchmark] Running CytoCommunity Algorithm...")
        adata, stats = run_cytocommunity_method(
            adata, 
            image_name="Benchmark_Image", 
            num_tcn=N_CLUSTERS, 
            knn_k=15, 
            num_run=5, 
            num_epoch=500
        )
        
        # 3. Report & Visualize
        calculate_metrics(adata, pred_key='pred_niche', true_key='niche', resource_stats=stats, emb_key='latent_embedding')
        plot_benchmark_results(adata, pred_key='pred_niche', true_key='niche', cell_type_key='cell_type', emb_key='latent_embedding')

        # 4. Save Results
        save_benchmark_results(adata, OUTPUT_PATH, method_name='CytoCommunity', pred_key='pred_niche', emb_key='latent_embedding')

        # 5. Save Time & Memory
        save_time_memory_results(stats, method_name='CytoCommunity', file_path=TIMEMEM_PATH)